In [1]:
import os, json, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

Device : cpu
PyTorch: 2.2.2+cpu


In [4]:
DATA_DIR   = Path("D:/PROJECTS/Speech_to_Emotions/emotion_data/processed")
OUTPUT_DIR = Path("D:/PROJECTS/Speech_to_Emotions/emotion_data/model_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load feature matrix and labels saved in the preprocessing notebook
X_raw = np.load(DATA_DIR / "X_features.npy")   # shape: (N, 438)
y_raw = np.load(DATA_DIR / "y_labels.npy")      # shape: (N,)

with open(DATA_DIR / "label_map.json") as f:
    label_map      = json.load(f)
    label_to_int   = label_map["label_to_int"]  # e.g. {"angry": 0, ...}
    int_to_label   = {int(k): v for k, v in label_map["int_to_label"].items()}

NUM_CLASSES  = len(label_to_int)
FEATURE_DIM  = X_raw.shape[1]   # 438

print(f"Total samples  : {len(X_raw)}")
print(f"Feature dim    : {FEATURE_DIM}")
print(f"Num classes    : {NUM_CLASSES}")
print(f"Label mapping  : {int_to_label}")

# Quick sanity check — no NaN or Inf should exist after preprocessing
assert not np.isnan(X_raw).any(), "NaN found in features!"
assert not np.isinf(X_raw).any(), "Inf found in features!"
print("\nData integrity: OK — no NaN or Inf values")

Total samples  : 9938
Feature dim    : 378
Num classes    : 6
Label mapping  : {0: 'angry', 1: 'disgust', 2: 'fear', 3: 'happy', 4: 'neutral', 5: 'sad'}

Data integrity: OK — no NaN or Inf values
